# Drought Forecast Verification — Template Notebook

This notebook demonstrates the full `drought-forecast-eval` workflow:

1. **Download** (optional) fresh data from BGDI
2. **Load** forecast archive and observation archive
3. **Align** observations to the forecast grid
4. **Build** a reference forecast (climatology or persistence)
5. **Compute** verification scores
6. **Visualise** skill curves and regional maps
7. **Export** results (NetCDF, CSV, HTML report)

Copy this notebook and adapt the paths / settings for your specific experiment.

---
**Docs:** `docs/` | **Source:** `drought_forecast_eval/` | **Issues / MRs:** GitLab repo

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

# ── EDIT THESE PATHS ──────────────────────────────────────────────────────
DATA_DIR = Path("data")  # root data dir

ARCHIVE_DIR = DATA_DIR / "zipfiles"  # flat dir of daily ZIPs
HISTORIC_ZIP = DATA_DIR / "historic.csv.zip"  # full observation archive
NC_DIR = DATA_DIR / "netcdf"  # netCDF dir
SHAPEFILE = Path("shapefiles/BAFU_Grossregionen_v3.shp")
OUTPUT_DIR = Path("output")

DATE_RANGE = ("2025-04-01", "2026-08-20")  # verification window
REGIONS = "all"  # or e.g. [1, 5, 12, 31]
VARIABLES = "all"  # or e.g. ["spi_1m", "cdi"]
CDI_THRESHOLD = 2  # CDI ≥ threshold = drought event
# ──────────────────────────────────────────────────────────────────────────

In [ ]:
! mkdir -p {NC_DIR}

## 1. (Optional) Download fresh data from BGDI

In [ ]:
from drought_forecast_eval.io.downloader import download_archive, download_historic

# Uncomment to download:
# download_archive(ARCHIVE_DIR, start=DATE_RANGE[0], end=DATE_RANGE[1])
# download_historic(HISTORIC_ZIP)

## 2. Load data

In [ ]:
import xarray as xr

from drought_forecast_eval.io.loader import load_forecasts, load_observations_combined

RECREATE_NETCDF = False  # set to True to re-create netCDF files from raw data

date_today = "2026-08-21"

if RECREATE_NETCDF:
    ds_fcst = load_forecasts(ARCHIVE_DIR, date_range=DATE_RANGE)
    ds_obs = load_observations_combined(HISTORIC_ZIP, archive_dir=ARCHIVE_DIR)
    ds_fcst.to_netcdf(NC_DIR / f"forecasts_{date_today}.nc")
    ds_obs.to_netcdf(NC_DIR / f"observations_{date_today}.nc")
else:
    ds_fcst = xr.open_dataset(NC_DIR / f"forecasts_{date_today}.nc")
    ds_obs = xr.open_dataset(NC_DIR / f"observations_{date_today}.nc")

In [ ]:
ds_fcst

In [ ]:
ds_obs

## 3. Align observations to the forecast grid

In [ ]:
from drought_forecast_eval import align_obs_to_forecast

if RECREATE_NETCDF:
    ds_merged = align_obs_to_forecast(ds_fcst, ds_obs)
    print("Merged dataset dims:", dict(ds_merged.dims))
    ds_merged.to_netcdf(NC_DIR / "merged.nc")
else:
    ds_merged = xr.open_dataset(NC_DIR / "merged.nc")

In [ ]:
ds_merged

In [ ]:
mask = ds_merged["forecast_reference_time"].dt.weekday == 0
ds_mondays = ds_merged.isel(forecast_reference_time=mask.values)

In [ ]:
ds_mondays

In [ ]:
ds_mondays.spi_1m_obs

## 4. Build reference forecast

Choose `"climatology"`, `"persistence"`, or `"both

In [ ]:
from drought_forecast_eval import climatology_reference, persistence_reference

CLIM_PERIOD = ("1991-01-01", "2020-12-31")  # WMO standard climate normal

if RECREATE_NETCDF:
    ds_ref_clim = climatology_reference(ds_obs, ds_fcst, week_window=1, valid_period=CLIM_PERIOD)
    ds_ref_pers = persistence_reference(ds_obs, ds_fcst)
    ds_ref_clim.to_netcdf(NC_DIR / "climatology_reference.nc")
    ds_ref_pers.to_netcdf(NC_DIR / "persistence_reference.nc")
else:
    ds_ref_clim = xr.open_dataset(NC_DIR / "climatology_reference.nc")
    ds_ref_pers = xr.open_dataset(NC_DIR / "persistence_reference.nc")


print("Climatology reference variables:", list(ds_ref_clim.data_vars)[:5], "...")

## 5 Explorative Data Analysis

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib.pylab as plt

%matplotlib inline
plt.show()

In [ ]:
RID = 42  # Region Identifier
VALIDATION_RANGE = slice("2026-01-01", "2026-08-20")

In [ ]:
from drought_forecast_eval.visualization.skill_curves import plot_forecast_plume

In [ ]:
ds_ref_pers

In [ ]:
fig, ax = plot_forecast_plume(
    ds_merged=ds_mondays.sel(forecast_reference_time=VALIDATION_RANGE),
    ds_ref_clim=ds_ref_clim,
    ds_ref_pers=ds_ref_pers,
    rid=42,
    var="soil_moisture_ufc",
    time_dim="forecast_reference_time",
    clim_dressed=True,
)

In [ ]:
fig, ax = plot_forecast_plume(
    ds_merged=ds_mondays.sel(forecast_reference_time=VALIDATION_RANGE),
    ds_ref_clim=ds_ref_clim,
    # ds_ref_pers=ds_ref_pers,
    rid=RID,
    var="precip_sum_1m",
)

In [ ]:
fig, ax = plot_forecast_plume(
    ds_merged=ds_mondays.sel(forecast_reference_time=slice("2026-06-08", "2026-06-15")),
    ds_ref_clim=ds_ref_clim,
    # ds_ref_pers=ds_ref_pers,
    rid=42,
    var="precip_sum_1m",
    clim_dressed=True,
)

In [ ]:
fig, ax = plot_forecast_plume(
    ds_merged=ds_mondays.sel(forecast_reference_time=slice("2026-01-01", "2026-08-20")),
    ds_ref_clim=ds_ref_clim,
    # ds_ref_pers=ds_ref_pers,
    rid=60,
    var="precip_sum_1m",
    clim_dressed=True,
)

In [ ]:
fig, ax = plot_forecast_plume(
    ds_merged=ds_mondays,
    ds_ref_clim=ds_ref_clim,
    # ds_ref_pers=ds_ref_pers,
    rid=42,
    var="cdi",
)

In [ ]:
fig, ax = plot_forecast_plume(
    ds_merged=ds_mondays,
    ds_ref_clim=ds_ref_clim,
    # ds_ref_pers=ds_ref_pers,
    rid=60,
    var="cdi",
)

## 6. Compute scores

In [ ]:
from drought_forecast_eval import compute_scores
import numpy as np

frt = ds_mondays["forecast_reference_time"]

SCORE_RANGE = slice("2026-03-01", "2026-07-15")
CDI_THRESHOLD = 4


ds_sel = ds_mondays.sel(forecast_reference_time=SCORE_RANGE)

ds_scores = compute_scores(
    ds_sel,
    variables=VARIABLES,
    scores=["bias", "correlation", "mae"],
    ds_ref=ds_ref_clim,
    cdi_threshold=CDI_THRESHOLD,
)

print("Score variables:", list(ds_scores.data_vars))

### 5a. Add custom metrics
You can register your own scoring function and include it by name:

In [ ]:
import xarray as xr
from drought_forecast_eval import register_score


def my_custom_score(fcst: xr.DataArray, obs: xr.DataArray, **kwargs) -> xr.DataArray:
    """Example: mean signed error."""
    return (fcst - obs).mean(dim=kwargs.get("reduce_dims", []))


register_score("signed_error", my_custom_score)

ds_scores_custom = compute_scores(
    ds_merged,
    variables=["spi_1m", "cdi"],
    scores=["mae", "signed_error"],
)
ds_scores_custom

## 6. Visualise results

In [ ]:
ds_scores

In [ ]:
import matplotlib.pyplot as plt
from drought_forecast_eval.visualization import plot_skill_vs_lead
from drought_forecast_eval.visualization.region_map import plot_region_polygon

In [ ]:
plot_region_polygon()

In [ ]:
for _score in ["correlation", "bias", "mae"]:
    fig = plot_skill_vs_lead(
        ds_scores.sel(drought_region_id=42),
        variables=[
            "precip_1m_index",
            "precip_3m_index",
            "precip_24m_index",
            "hydro_index",
            "soil_moisture_index",
            "cdi",
        ],
        aggregate_regions=True,
        score=f"{_score}",
        title=f"{_score} by lead week",
    )
    plt.show()

In [ ]:
for _score in ["correlation", "bias", "mae"]:
    fig = plot_skill_vs_lead(
        ds_scores.sel(drought_region_id=60),
        variables=[
            "precip_1m_index",
            "precip_3m_index",
            "precip_24m_index",
            "hydro_index",
            "soil_moisture_index",
            "cdi",
        ],
        aggregate_regions=True,
        score=f"{_score}",
        title=f"{_score} by lead week",
    )
    plt.show()

In [ ]:
for _score in ["correlation", "bias", "mae"]:
    fig = plot_skill_vs_lead(
        ds_scores,  # .sel(drought_region_id=60),
        variables=[
            "precip_1m_index",
            "precip_3m_index",
            "precip_24m_index",
            "hydro_index",
            "soil_moisture_index",
            "cdi",
        ],
        aggregate_regions=True,
        score=f"{_score}",
        title=f"{_score} by lead week",
    )
    plt.show()

In [ ]:
ds_scores

In [ ]:
# Regional map (requires shapefile; falls back to bar chart if not provided)
from drought_forecast_eval.visualization import plot_region_map


score = "bias"

_vmin, _vmax = -1.5, 1.5

fig3 = plot_region_map(
    ds_scores=ds_scores,
    score_variable=f"precip_1m_index_{score}",
    shapefile_path=SHAPEFILE,
    lead_weeks=[0, 1, 2, 3],
    cmap="BrBG_r",
    vmin=_vmin,
    vmax=_vmax,
)
fig3 = plot_region_map(
    ds_scores=ds_scores,
    score_variable=f"precip_3m_index_{score}",
    shapefile_path=SHAPEFILE,
    lead_weeks=[0, 1, 2, 3],
    cmap="BrBG_r",
    vmin=-1,
    vmax=1,
)
fig3 = plot_region_map(
    ds_scores=ds_scores,
    score_variable=f"precip_24m_index_{score}",
    shapefile_path=SHAPEFILE,
    lead_weeks=[0, 1, 2, 3],
    cmap="BrBG_r",
    vmin=_vmin,
    vmax=_vmax,
)
fig4 = plot_region_map(
    ds_scores=ds_scores,
    score_variable=f"hydro_index_{score}",
    shapefile_path=SHAPEFILE,
    lead_weeks=[0, 1, 2, 3],
    cmap="BrBG_r",
    vmin=_vmin,
    vmax=_vmax,
)
fig4 = plot_region_map(
    ds_scores=ds_scores,
    score_variable=f"soil_moisture_index_{score}",
    shapefile_path=SHAPEFILE,
    lead_weeks=[0, 1, 2, 3],
    cmap="BrBG_r",
    vmin=_vmin,
    vmax=_vmax,
)


fig5 = plot_region_map(
    ds_scores=ds_scores,
    score_variable=f"cdi_{score}",
    shapefile_path=SHAPEFILE,
    lead_weeks=[0, 1, 2, 3],
    cmap="BrBG_r",
    vmin=_vmin,
    vmax=_vmax,
)


plt.show()

In [ ]:
# Regional map (requires shapefile; falls back to bar chart if not provided)
from drought_forecast_eval.visualization import plot_region_map


score = "correlation"

_vmin, _vmax = 0, 1

cmap = "plasma"

fig3 = plot_region_map(
    ds_scores=ds_scores,
    score_variable=f"precip_1m_index_{score}",
    shapefile_path=SHAPEFILE,
    lead_weeks=[0, 1, 2, 3],
    cmap=cmap,
    vmin=_vmin,
    vmax=_vmax,
)
fig3 = plot_region_map(
    ds_scores=ds_scores,
    score_variable=f"precip_3m_index_{score}",
    shapefile_path=SHAPEFILE,
    lead_weeks=[0, 1, 2, 3],
    cmap=cmap,
    vmin=_vmin,
    vmax=1,
)
fig3 = plot_region_map(
    ds_scores=ds_scores,
    score_variable=f"precip_24m_index_{score}",
    shapefile_path=SHAPEFILE,
    lead_weeks=[0, 1, 2, 3],
    cmap=cmap,
    vmin=_vmin,
    vmax=_vmax,
)
fig4 = plot_region_map(
    ds_scores=ds_scores,
    score_variable=f"hydro_index_{score}",
    shapefile_path=SHAPEFILE,
    lead_weeks=[0, 1, 2, 3],
    cmap=cmap,
    vmin=_vmin,
    vmax=_vmax,
)
fig4 = plot_region_map(
    ds_scores=ds_scores,
    score_variable=f"soil_moisture_index_{score}",
    shapefile_path=SHAPEFILE,
    lead_weeks=[0, 1, 2, 3],
    cmap=cmap,
    vmin=_vmin,
    vmax=_vmax,
)


fig4 = plot_region_map(
    ds_scores=ds_scores,
    score_variable=f"cdi_{score}",
    shapefile_path=SHAPEFILE,
    lead_weeks=[0, 1, 2, 3],
    cmap=cmap,
    vmin=_vmin,
    vmax=_vmax,
)


plt.show()

In [ ]:
ds_ref_clim

In [ ]:
## Comparison of experiments

In [ ]:
# Experiment comparison: period A vs period B
from drought_forecast_eval.visualization import plot_experiment_diff

# Example: compare first half vs second half of the year
DATE_RANGE_A = ("2025-07-01", "2025-10-29")
DATE_RANGE_B = ("2026-03-01", "2026-07-30")

ds_merged_a = ds_merged.sel(forecast_reference_time=slice(*DATE_RANGE_A))
ds_merged_b = ds_merged.sel(forecast_reference_time=slice(*DATE_RANGE_B))

cdi_vars = [
    "precip_1m_index",
    "precip_3m_index",
    "precip_24m_index",
    "hydro_index",
    "soil_moisture_index",
    "cdi",
]
scores = ["mae", "correlation", "bias"]

ds_scores_a = compute_scores(ds_merged_a, variables=cdi_vars, scores=scores, ds_ref=ds_ref_clim)
ds_scores_b = compute_scores(ds_merged_b, variables=cdi_vars, scores=scores, ds_ref=ds_ref_clim)

fig4 = plot_experiment_diff(
    ds_scores_a,
    ds_scores_b,
    variables=cdi_vars,
    score="mae",
    label_a="summer 2025",
    label_b="summer 2026",
)
plt.show()

## 7. Export results

In [ ]:
from drought_forecast_eval import ReportBuilder

builder = ReportBuilder(
    archive_dir=ARCHIVE_DIR,
    historic_path=HISTORIC_ZIP,
    date_range=DATE_RANGE,
    regions=REGIONS,
    variables=VARIABLES,
    scores="default",
    reference="climatology",
    cdi_threshold=CDI_THRESHOLD,
    shapefile_path=SHAPEFILE,
    output_dir=OUTPUT_DIR,
)

# Run the full pipeline and write all outputs
outputs = builder.run()
print("Outputs:")
for key, path in outputs.items():
    print(f"  {key}: {path}")

In [ ]:
# Or export individually:
# builder.export_netcdf(OUTPUT_DIR / "scores.nc")
# builder.export_csv(OUTPUT_DIR / "scores_summary.csv")
# builder.export_html(OUTPUT_DIR / "report.html")

In [ ]:
def _exceedance_prob_from_quantiles(
    p10_arr: np.ndarray,
    p50_arr: np.ndarray,
    p90_arr: np.ndarray,
    threshold: float,
    above: bool = True,
) -> np.ndarray:
    """Compute P(forecast >= threshold) from a 3-quantile forecast.
    If above is False P(forecast <= threshold) form a 3-quantile forecast is
    computed.

    Uses a piecewise-linear CDF with anchors:
        (−∞ → 0,  p10 → 0.1,  p50 → 0.5,  p90 → 0.9,  +∞ → 1)

    Returns 0 below p10 and 1 above p90.
    Set ``above=False`` for subceedance: P(forecast <= threshold).
    """
    denom_low = np.where(p50_arr > p10_arr, p50_arr - p10_arr, 1e-10)
    denom_high = np.where(p90_arr > p50_arr, p90_arr - p50_arr, 1e-10)

    cdf = np.where(
        threshold <= p10_arr,
        0.0,
        np.where(
            threshold <= p50_arr,
            0.1 + 0.4 * (threshold - p10_arr) / denom_low,
            np.where(
                threshold >= p90_arr,
                0.5 + 0.4 * (threshold - p50_arr) / denom_high,
                1.0,
            ),
        ),
    )
    return 1.0 - cdf if above else cdf


def _build_auc_da(
    obs: xr.DataArray,
    fcst_p50: xr.DataArray,
    threshold: float,
    above: bool,
    reduce_dims: list[str],
    fcst_p10: xr.DataArray | None,
    fcst_p90: xr.DataArray | None,
    score_fn,
    long_name: str,
) -> xr.DataArray:
    """Shared loop for ROC-AUC and PR-AUC."""
    use_exceedance = fcst_p10 is not None and fcst_p90 is not None

    def _ranking_score(p50_flat, p10_flat, p90_flat):
        if use_exceedance:
            return _exceedance_prob_from_quantiles(
                p10_flat, p50_flat, p90_flat, threshold, above=above
            )
        return p50_flat if above else -p50_flat

    def _obs_bin(arr):
        return (arr >= threshold).astype(int) if above else (arr <= threshold).astype(int)

    keep_dims = [d for d in obs.dims if d not in reduce_dims]

    if not keep_dims:
        obs_float = obs.values.ravel().astype(float)
        obs_b = _obs_bin(obs_float)
        p50_flat = fcst_p50.values.ravel().astype(float)
        p10_flat = fcst_p10.values.ravel().astype(float) if use_exceedance else None
        p90_flat = fcst_p90.values.ravel().astype(float) if use_exceedance else None
        score = _ranking_score(p50_flat, p10_flat, p90_flat)
        valid = np.isfinite(obs_float) & np.isfinite(score)
        if valid.sum() < 2 or obs_b[valid].sum() == 0 or obs_b[valid].sum() == valid.sum():
            return xr.DataArray(np.nan, attrs={"long_name": long_name})
        return xr.DataArray(
            score_fn(obs_b[valid], score[valid]),
            attrs={"long_name": long_name, "threshold": threshold},
        )

    results, coords_list = [], []
    first_keep_dim = keep_dims[0]
    for idx in obs[first_keep_dim].values:
        sel = {first_keep_dim: idx}
        obs_s = obs.sel(sel).values.ravel().astype(float)
        p50_s = fcst_p50.sel(sel).values.ravel().astype(float)
        p10_s = fcst_p10.sel(sel).values.ravel().astype(float) if use_exceedance else None
        p90_s = fcst_p90.sel(sel).values.ravel().astype(float) if use_exceedance else None
        score_s = _ranking_score(p50_s, p10_s, p90_s)
        obs_b = _obs_bin(obs_s)
        valid = np.isfinite(obs_s) & np.isfinite(score_s)
        if valid.sum() < 2 or obs_b[valid].sum() == 0 or obs_b[valid].sum() == valid.sum():
            auc = np.nan
        else:
            try:
                auc = score_fn(obs_b[valid], score_s[valid])
            except ValueError:
                auc = np.nan
        results.append(auc)
        coords_list.append(idx)

    return xr.DataArray(
        results,
        coords={first_keep_dim: coords_list},
        dims=[first_keep_dim],
        attrs={"long_name": long_name, "threshold": threshold},
    )


def roc_auc_xr(
    obs: xr.DataArray,
    fcst_p50: xr.DataArray,
    threshold: float = 2.0,
    above: bool = True,
    reduce_dims: list[str] | None = None,
    fcst_p10: xr.DataArray | None = None,
    fcst_p90: xr.DataArray | None = None,
) -> xr.DataArray:
    """Compute ROC-AUC for a binary event (obs >= threshold if above, else obs <= threshold).

    See ``_build_auc_da`` for parameter details.
    """
    from sklearn.metrics import roc_auc_score

    if reduce_dims is None:
        reduce_dims = ["forecast_reference_time", "drought_region_id"]

    return _build_auc_da(
        obs, fcst_p50, threshold, above, reduce_dims, fcst_p10, fcst_p90, roc_auc_score, "ROC-AUC"
    )


def pr_auc_xr(
    obs: xr.DataArray,
    fcst_p50: xr.DataArray,
    threshold: float = 2.0,
    above: bool = True,
    reduce_dims: list[str] | None = None,
    fcst_p10: xr.DataArray | None = None,
    fcst_p90: xr.DataArray | None = None,
) -> xr.DataArray:
    """Compute Precision-Recall AUC for a binary event (obs >= threshold if above, else obs <= threshold).

    Uses ``average_precision_score`` (area under the PR curve via trapezoidal rule).
    See ``_build_auc_da`` for parameter details.
    """
    from sklearn.metrics import average_precision_score

    if reduce_dims is None:
        reduce_dims = ["forecast_reference_time", "drought_region_id"]

    return _build_auc_da(
        obs,
        fcst_p50,
        threshold,
        above,
        reduce_dims,
        fcst_p10,
        fcst_p90,
        average_precision_score,
        "PR-AUC",
    )

In [ ]:
_ds_merged = ds_merged.sel(drought_region_id=40)

In [ ]:
for t in [-2, -1, -0.5, 0]:
    roc_auc_xr(
        _ds_merged.spi_1m_obs,
        fcst_p50=_ds_merged.spi_1m_p50,
        threshold=t,
        fcst_p10=_ds_merged.spi_1m_p10,
        fcst_p90=_ds_merged.spi_1m_p90,
        above=False,
    ).plot(label=f"{t}")
    plt.legend(title="spi1m below")

In [ ]:
from scipy.stats import norm

thresholds = [-2, -1, -0.5, 0]
for t, color in zip(thresholds, plt.cm.tab10.colors):
    pr_auc_xr(
        _ds_merged.spi_1m_obs,
        fcst_p50=_ds_merged.spi_1m_p50,
        threshold=t,
        fcst_p10=_ds_merged.spi_1m_p10,
        fcst_p90=_ds_merged.spi_1m_p90,
        above=False,
    ).plot(label=f"{t}", color=color)
    # baseline = prevalence P(N(0,1) <= t) for a random classifier
    plt.axhline(norm.cdf(t), color=color, linestyle="--", linewidth=0.8)

plt.legend(
    title="spi1m below\n(-- = baseline)",
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    borderaxespad=0,
)

In [ ]:
for t in [-2, -1, -0.5, 0]:
    roc_auc_xr(
        _ds_merged.spi_12m_obs,
        fcst_p50=_ds_merged.spi_12m_p50,
        threshold=t,
        fcst_p10=_ds_merged.spi_12m_p10,
        fcst_p90=_ds_merged.spi_1m_p90,
        above=False,
    ).plot(label=f"{t}")
    plt.legend(title="spi1m below")

In [ ]:
for t in [-1, -0.5, 0]:
    roc_auc_xr(
        _ds_merged.spi_24m_obs,
        fcst_p50=_ds_merged.spi_24m_p50,
        threshold=t,
        fcst_p10=_ds_merged.spi_24m_p10,
        fcst_p90=_ds_merged.spi_1m_p90,
        above=False,
    ).plot(label=f"{t}")
    plt.legend(title="spi1m below")

In [ ]:
_exceedance_prob_from_quantiles(
    ds_merged.spi_1m_p10, ds_merged.spi_1m_p50, ds_merged.spi_1m_p90, threshold=-1
)

In [ ]:
roc_auc_xr(
    ds_mondays.spi_1m_obs,
    fcst_p50=ds_mondays.spi_1m_p50,
    threshold=-1,
    fcst_p10=ds_mondays.spi_1m_p10,
    fcst_p90=ds_mondays.spi_1m_p90,
)